In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/xux/Desktop/AlphaVariant/Benchmark

# AlphaVariant per-round improvement trajectories (panel e)

Shows how AlphaVariant's best-seen normalised fitness improves round by round on the three multi-site oracle benchmarks: **AAV**, **CreiLOV**, and **PAB1**.

For each dataset, all 30 independent runs are plotted as thin lines (α ≈ 0.3).  
The Q1–Q3 band (IQR) is shaded, and the median trajectory is drawn as a thicker vermilion line with dots.

Data source: `results_oracle/{dataset}/AlphaVariant/seed*.json` → `metrics.fitness_trajectory`  
(5-element list, one value per optimisation round, already normalised to [0, 1]).

Outputs (PNG + PDF + SVG) → `figures/ms_oracles/`.

In [ ]:
import os, sys, glob, json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, '.')
from utils.plot_style_utils import VERMILION, apply_nature_rcparams, save_figure

OUTDIR = os.path.join('figures', 'ms_oracles')
os.makedirs(OUTDIR, exist_ok=True)

# Salmon / light-pink for individual run lines — slightly desaturated vermilion
SALMON  = '#F4A07A'
IQR_BG  = '#F8CBB0'    # very light fill for the IQR band

apply_nature_rcparams()   # all defaults are correct for this figure

print('Setup complete.')

## Load trajectory data

In [ ]:
DATASETS = {
    'ms_AAV':     'AAV',
    'ms_CreiLOV': 'CreiLOV',
    'ms_PAB1':    'PAB1',
}
N_ROUNDS = 5


def load_trajectories(dataset_key, cap=30):
    """Return an array of shape (n_seeds, N_ROUNDS) from fitness_trajectory in each seed JSON."""
    pattern = f'results_oracle/{dataset_key}/AlphaVariant/seed*.json'
    trajs = []
    for fp in sorted(glob.glob(pattern))[:cap]:
        try:
            d = json.load(open(fp))
            traj = d.get('metrics', {}).get('fitness_trajectory')
            if traj is None:
                traj = d.get('fitness_trajectory')
            if traj is not None and len(traj) == N_ROUNDS:
                trajs.append([float(v) for v in traj])
        except Exception:
            pass
    if not trajs:
        return None
    return np.array(trajs)   # shape: (n_seeds, N_ROUNDS)


data = {}
for key, label in DATASETS.items():
    arr = load_trajectories(key)
    if arr is not None:
        data[key] = arr
        print(f'{label:12s}: {arr.shape[0]} seeds loaded')
    else:
        print(f'{label:12s}: NO DATA FOUND at results_oracle/{key}/AlphaVariant/seed*.json')

## Plot trajectory figure

In [ ]:
def plot_trajectory_figure(data, save=True):
    rounds = np.arange(1, N_ROUNDS + 1)
    n_panels = len(data)

    fig, axes = plt.subplots(1, n_panels,
                             figsize=(2.6 * n_panels + 0.7, 2.8),
                             sharey=False)
    axes = np.atleast_1d(axes)
    fig.patch.set_facecolor('white')

    for ax, (key, arr) in zip(axes, data.items()):
        label = DATASETS[key]
        n_seeds = arr.shape[0]

        # --- thin individual run lines ---
        for seed_row in arr:
            ax.plot(rounds, seed_row, color=SALMON, lw=0.7, alpha=0.28, zorder=2)

        # --- IQR band ---
        q1  = np.percentile(arr, 25, axis=0)
        q3  = np.percentile(arr, 75, axis=0)
        med = np.median(arr, axis=0)
        ax.fill_between(rounds, q1, q3, color=IQR_BG, alpha=0.65, zorder=3, linewidth=0)
        ax.plot(rounds, q1, color=SALMON, lw=0.4, alpha=0.55, zorder=3)
        ax.plot(rounds, q3, color=SALMON, lw=0.4, alpha=0.55, zorder=3)

        # --- median trajectory ---
        ax.plot(rounds, med, color=VERMILION, lw=2.0, zorder=5, solid_capstyle='round')
        ax.scatter(rounds, med, s=28, color=VERMILION, zorder=6, edgecolors='white', linewidths=0.6)

        # Axis styling
        ax.set_title(label, fontweight='bold', pad=6)
        ax.set_xlabel('Optimisation round', labelpad=3)
        ax.set_xlim(0.6, N_ROUNDS + 0.4)
        ax.set_xticks(rounds)
        ax.set_ylim(bottom=max(0.0, arr.min() - 0.06))
        ax.yaxis.grid(True, color='#E6E6E6', lw=0.55, zorder=0)
        ax.xaxis.grid(False)
        ax.tick_params(axis='both', length=2.4, width=0.55, color='#333333', pad=2)
        for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
        for sp in ['left', 'bottom']:
            ax.spines[sp].set_color('#333333')
            ax.spines[sp].set_linewidth(0.6)

    axes[0].set_ylabel('Best-seen normalised fitness', labelpad=3)

    # Shared legend
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    legend_handles = [
        Line2D([0], [0], color=SALMON, lw=1.2, alpha=0.55,
               label=f'Individual run (n = {list(data.values())[0].shape[0]})'),
        Patch(facecolor=IQR_BG, edgecolor=SALMON, lw=0.5, label='Q1–Q3 (IQR)'),
        Line2D([0], [0], color=VERMILION, lw=2.0,
               marker='o', markersize=5.5, markerfacecolor=VERMILION,
               markeredgecolor='white', markeredgewidth=0.6, label='Median'),
    ]
    fig.legend(handles=legend_handles, loc='lower center',
               bbox_to_anchor=(0.5, -0.01), ncol=3, frameon=False,
               fontsize=6.2, handletextpad=0.5, columnspacing=1.2)

    fig.suptitle('AlphaVariant per-round fitness improvement on multi-site benchmarks',
                 x=0.04, y=1.02, ha='left', va='bottom',
                 fontsize=9.8, fontweight='bold')
    fig.text(
        0.04, -0.12,
        'Thin lines: individual runs; shaded band: Q1–Q3 (IQR); '
        'thick line with dots: median. Fitness normalised to [0, 1] via oracle model.',
        ha='left', va='top', fontsize=5.6, color='#4D4D4D',
    )
    fig.subplots_adjust(left=0.10, right=0.985, bottom=0.22, top=0.88, wspace=0.35)

    if save:
        save_figure(fig, OUTDIR, 'alphavariant_trajectory_multisite')

    plt.show()
    return fig


print('plot_trajectory_figure() defined.')

## Generate figure

In [ ]:
if data:
    fig = plot_trajectory_figure(data)
else:
    print('No trajectory data found — check results_oracle/ paths above.')

## Quick summary statistics

In [ ]:
import pandas as pd

rows = []
for key, arr in data.items():
    label = DATASETS[key]
    for r_idx, round_no in enumerate(range(1, N_ROUNDS + 1)):
        col = arr[:, r_idx]
        rows.append({
            'dataset': label,
            'round': round_no,
            'n': len(col),
            'median': round(np.median(col), 4),
            'q1': round(np.percentile(col, 25), 4),
            'q3': round(np.percentile(col, 75), 4),
            'min': round(col.min(), 4),
            'max': round(col.max(), 4),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

csv_path = os.path.join(OUTDIR, 'alphavariant_trajectory_summary.csv')
summary.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')